**PySpark Join Types (Point-wise)**


**Inner Join (inner)****

Returns only matching records from both DataFrames.

**Left Outer Join (left, leftouter, left_outer)**

Returns all records from the left DataFrame and matching records from the right DataFrame.

**Right Outer Join (right, rightouter, right_outer)**

Returns all records from the right DataFrame and matching records from the left DataFrame.

**Full Outer Join (full, outer, fullouter, full_outer)**

Returns all records from both DataFrames, including unmatched records.

**Left Semi Join (semi, leftsemi, left_semi)**

Returns only matching rows from the left DataFrame.
Columns from the right DataFrame are not included.

**Left Anti Join (anti, leftanti, left_anti)**

Returns only non-matching rows from the left DataFrame.

**Cross Join (cross)**

Returns the Cartesian product of both DataFrames.
Every row from the left is combined with every row from the right.

**Self Join**

Joins a DataFrame with itself using aliases.
Commonly used to find hierarchical relationships (Employee-Manager, Parent-Child, etc.).
No separate join type exists; uses inner, left, right, etc., on the same DataFrame.

In [0]:
from pyspark.sql.functions import col
#Employee dataframe
employee_data = [
    (10, "Raj", "1999", "100", "M", 2000),
    (20, "Rahul", "2002", "200", "M", 8000),
    (30, "Raghav", "2010", "100", "M", 6000),
    (40, "Raja", "2004", "100", "F", 7000),
    (50, "Rama", "2008", "400", "F", 5000),
    (60, "Rasul", "2014", "500", "M", 5000)
]
employee_schema = [
    "employee_id", "name", "doj",
    "employee_dept_id", "gender", "salary"
]
employeeDF = spark.createDataFrame(data=employee_data, schema=employee_schema)

#create department dataframe
department_data = [
    ("HR", 100),
    ("Supply", 200),
    ("Sales", 300),
    ("Stock", 400)
]
department_schema = ["dept_name", "dept_id"]
departmentDF = spark.createDataFrame(data=department_data, schema=department_schema)


different joins

In [0]:

#inner join
employeeDF.join(departmentDF, employeeDF.employee_dept_id == departmentDF.dept_id, 'inner').show()

#left join
employeeDF.join(departmentDF, employeeDF.employee_dept_id == departmentDF.dept_id, 'left').show()

#right join
employeeDF.join(departmentDF, employeeDF.employee_dept_id == departmentDF.dept_id, 'right').show()

#full join
employeeDF.join(departmentDF, employeeDF.employee_dept_id == departmentDF.dept_id, 'full').show()

#Giving matching records from left table (left semi)
df_join = employeeDF.join(departmentDF, employeeDF.employee_dept_id == departmentDF.dept_id, "left_semi")
display(df_join)

#Unmatched records from left table (anti)
df_join = employeeDF.join(departmentDF, employeeDF.employee_dept_id == departmentDF.dept_id, "anti")
display(df_join)

In [0]:
#self join-
# Add a manager_id column referencing employee_id for the self join demo
from pyspark.sql.functions import when, lit

employeeDFWithMgr = employeeDF.withColumn(
    "manager_id",
    when(col("employee_id") == 10, lit(None).cast("int"))
    .when(col("employee_id") == 20, lit(10))
    .when(col("employee_id") == 30, lit(10))
    .when(col("employee_id") == 40, lit(20))
    .when(col("employee_id") == 50, lit(20))
    .when(col("employee_id") == 60, lit(30))
)

employeeDFWithMgr.alias('empdata') \
  .join(employeeDFWithMgr.alias('mgrdata'), col('empdata.manager_id') == col('mgrdata.employee_id'), 'left') \
  .select(
      col('empdata.name').alias('empName'),
      col('mgrdata.name').alias('managerName')
  ).show()